# Titler v1 — from-scratch tiny T5 training (Colab free T4)

Before running:
1. Locally: run stages 01-05 + `train_tokenizer.py` so you have `train.jsonl`, `val.jsonl`, and `model/tokenizer/`.
2. Upload to Google Drive under `titler/`: `train.jsonl`, `val.jsonl`, and the `tokenizer/` folder.
3. Runtime → Change runtime type → T4 GPU. Run all.
4. Download `titler-v1.zip` from Drive when done → unzip to `model/titler-v1/` in the repo.

In [ ]:
# ---- parameters ----
DRIVE_DIR = '/content/drive/MyDrive/titler'
VOCAB_SIZE = 8192
MAX_INPUT_LEN, MAX_TARGET_LEN = 512, 16
LR = 5e-4
EPOCHS = 10
BATCH_SIZE = 128
WARMUP_STEPS = 2000
LABEL_SMOOTHING = 0.1

In [ ]:
%pip install -q transformers datasets rouge-score accelerate
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU, then Run all again.'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, numpy as np
from datasets import Dataset
from transformers import (AutoTokenizer, T5Config, T5ForConditionalGeneration,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          GenerationConfig)
from rouge_score import rouge_scorer

tokenizer = AutoTokenizer.from_pretrained(f'{DRIVE_DIR}/tokenizer')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return Dataset.from_list([json.loads(l) for l in f if l.strip()])

train_ds, val_ds = load_jsonl(f'{DRIVE_DIR}/train.jsonl'), load_jsonl(f'{DRIVE_DIR}/val.jsonl')

def preprocess(batch):
    enc = tokenizer(batch['prompt'], truncation=True, max_length=MAX_INPUT_LEN)
    labels = tokenizer(batch['title'], truncation=True, max_length=MAX_TARGET_LEN)
    enc['labels'] = labels['input_ids']
    return enc

cols = train_ds.column_names
train_tok = train_ds.map(preprocess, batched=True, remove_columns=cols)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=cols)
print(len(train_tok), len(val_tok))

In [ ]:
config = T5Config(
    vocab_size=VOCAB_SIZE, d_model=256, d_kv=64, num_heads=4, d_ff=1024,
    num_layers=3, num_decoder_layers=3, feed_forward_proj='relu',
    dropout_rate=0.1, relative_attention_num_buckets=32, relative_attention_max_distance=128,
    tie_word_embeddings=True, decoder_start_token_id=0, pad_token_id=0, eos_token_id=1)
model = T5ForConditionalGeneration(config)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M params')

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    pred_txt = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_txt = tokenizer.batch_decode(labels, skip_special_tokens=True)
    scores = [scorer.score(l, p)['rougeL'].fmeasure for l, p in zip(label_txt, pred_txt)]
    return {'rougeL': float(np.mean(scores))}

args = Seq2SeqTrainingArguments(
    output_dir=f'{DRIVE_DIR}/checkpoints',
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR, lr_scheduler_type='cosine', warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS, label_smoothing_factor=LABEL_SMOOTHING, fp16=True,
    eval_strategy='steps', eval_steps=1000, save_steps=1000, save_total_limit=2,
    predict_with_generate=True, generation_max_length=MAX_TARGET_LEN,
    load_best_model_at_end=True, metric_for_best_model='rougeL',
    logging_steps=100, report_to='none')

trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics)
trainer.train()

In [ ]:
out = f'{DRIVE_DIR}/titler-v1'
trainer.save_model(out)
tokenizer.save_pretrained(out)
GenerationConfig(decoder_start_token_id=0, pad_token_id=0, eos_token_id=1,
                 max_new_tokens=16, num_beams=4, no_repeat_ngram_size=2).save_pretrained(out)

# quick smoke
import torch
model.eval()
for p in ['how do I center a div with flexbox', 'write a poem about the ocean', 'hi']:
    ids = tokenizer(p, return_tensors='pt').to(model.device)
    print(p, '->', tokenizer.decode(model.generate(**ids, max_new_tokens=16, num_beams=4)[0], skip_special_tokens=True))

!cd {DRIVE_DIR} && zip -rq titler-v1.zip titler-v1
print('download', f'{DRIVE_DIR}/titler-v1.zip')